# Inspecting the persisted data model

Read the original small Spark output with DuckDB. These are native queries over Parquet, not a newly deployed SQL warehouse. The `day2-final` fixture includes a machine change; the Airflow pilot is a separate dataset. No SQL foreign-key constraint is claimed for Parquet.

Run from a repository with `data/lakehouse/day2-final` restored. Requires DuckDB 1.4.3 and pandas. The original files are hashed before and after the queries; only the notebook's outputs are saved.

In [ ]:
from pathlib import Path
from hashlib import sha256
import json
import duckdb
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'pipelines/transforms.py').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Open this notebook from the project checkout')
    ROOT = ROOT.parent
SOURCE = ROOT / 'data/lakehouse/day2-final'
files = sorted(p for zone in ['bronze', 'silver', 'gold'] for p in (SOURCE / zone).rglob('*.parquet'))
assert files, 'Restore the original small Spark output first; do not silently use a different cohort'
before = {str(p.relative_to(SOURCE)): sha256(p.read_bytes()).hexdigest() for p in files}
con = duckdb.connect(':memory:', config={'threads': 2, 'memory_limit': '1GB'})
inventory = []
for zone in ['bronze', 'silver', 'gold']:
    con.execute(f'CREATE SCHEMA {zone}')
    for folder in sorted((SOURCE / zone).glob('*')):
        parts = sorted(folder.glob('*.parquet')) if folder.is_dir() else []
        if not parts:
            continue
        paths = '[' + ','.join("'" + p.as_posix().replace("'", "''") + "'" for p in parts) + ']'
        con.execute(f'CREATE VIEW {zone}.{folder.name} AS SELECT * FROM read_parquet({paths})')
        rows = con.execute(f'SELECT count(*) FROM {zone}.{folder.name}').fetchone()[0]
        inventory.append({'zone': zone, 'table': folder.name, 'rows': rows})
print(f'DuckDB {duckdb.__version__} | Original Spark small fixture | {len(files)} Parquet files')
pd.DataFrame(inventory)

## Duplicate exports and schema evolution

Bronze retains original source envelopes. Silver resolves repeated event IDs and parses the declared union schema. Missing v1 pressure means unknown, not zero. This is a correctness comparison, not an isolated performance benchmark.

In [ ]:
display(con.sql("""
SELECT 'Bronze telemetry exports' AS stage, count(*) AS rows,
       count(DISTINCT json_extract_string(payload, '$.event_id')) AS unique_events
FROM bronze.raw_department_records WHERE source_path LIKE 'telemetry/%'
UNION ALL
SELECT 'Silver normalized readings', count(*), count(DISTINCT event_id)
FROM silver.stg_telemetry
""").df())
display(con.sql("""
SELECT schema_version, count(*) AS unique_readings,
       count(pressure_bar) AS pressure_present,
       count(*) - count(pressure_bar) AS pressure_unknown,
       typeof(any_value(pressure_bar)) AS normalized_type
FROM silver.stg_telemetry GROUP BY schema_version ORDER BY schema_version
""").df())

## Historical machine versions

`machine_sk` identifies a version; `machine_id` identifies the physical machine. The event-time join uses a half-open interval. P-001 moves from F-01 to F-04 on 2 October. These analytics joins are separate from prediction-time availability checks.

In [ ]:
display(con.sql("""
SELECT machine_id, factory_id, valid_from_ts, valid_to_ts, is_current, recorded_ts
FROM gold.dim_machine WHERE machine_id='P-001' ORDER BY valid_from_ts
""").df())
display(con.sql("""
SELECT count(*) AS versions, count(DISTINCT machine_sk) AS distinct_version_keys,
       count(DISTINCT machine_id) AS machines, count(*) FILTER (WHERE is_current) AS current_versions
FROM gold.dim_machine
""").df())
display(con.sql("""
SELECT CAST(event_ts AS DATE) AS event_date, factory_id, count(*) AS readings
FROM gold.fact_telemetry WHERE machine_id='P-001'
GROUP BY event_date, factory_id ORDER BY event_date
""").df())

## Fact/dimension relationships

Each fact version key must resolve exactly one machine version; each dimension factory must resolve a factory. The pipeline validates these relationships. The following queries inspect persisted results without adding artificial database constraints.

In [ ]:
display(con.sql("""
SELECT 'fact_telemetry -> dim_machine' AS relationship, count(*) AS fact_rows,
       count(*) FILTER (WHERE d.machine_sk IS NULL) AS missing_parent
FROM gold.fact_telemetry f LEFT JOIN gold.dim_machine d USING(machine_sk)
UNION ALL
SELECT 'fact_failure -> dim_machine', count(*), count(*) FILTER (WHERE d.machine_sk IS NULL)
FROM gold.fact_failure f LEFT JOIN gold.dim_machine d USING(machine_sk)
UNION ALL
SELECT 'fact_maintenance -> dim_machine', count(*), count(*) FILTER (WHERE d.machine_sk IS NULL)
FROM gold.fact_maintenance f LEFT JOIN gold.dim_machine d USING(machine_sk)
UNION ALL
SELECT 'dim_machine -> dim_factory', count(*), count(*) FILTER (WHERE d.factory_id IS NULL)
FROM gold.dim_machine m LEFT JOIN gold.dim_factory d USING(factory_id)
""").df())

## Feature timestamps and input schema

One row is a `(machine_id, event_timestamp)` snapshot. `created` is build-audit time; `max_feature_event_ts` and `max_received_ts` describe available input history. Future labels remain a separate table. Inspecting these columns alone does not replace the pipeline's leakage tests.

In [ ]:
display(con.sql('DESCRIBE gold.feat_machine_10m').df()[['column_name', 'column_type']])
display(con.sql("""
SELECT machine_id, event_timestamp, created, max_feature_event_ts, max_received_ts
FROM gold.feat_machine_10m ORDER BY machine_id, event_timestamp LIMIT 4
""").df())
display(con.sql("""
SELECT count(*) AS snapshots, count(DISTINCT (machine_id,event_timestamp)) AS unique_snapshots,
       count(*) FILTER (WHERE max_feature_event_ts > event_timestamp OR max_received_ts > event_timestamp) AS future_input_rows
FROM gold.feat_machine_10m
""").df())
after = {str(p.relative_to(SOURCE)): sha256(p.read_bytes()).hexdigest() for p in files}
assert before == after
print('PASS: all original Parquet bytes unchanged')